# 01 · Preserve the tensor identity and freeze the patient split

Continue from the existing completed monthly `tensor_complete` table. The shared reference shows a Spark DataFrame with `PATIENT_ID`, `END_DT`, `RESP`, `TIME_STEP` and the established `RX__`/`DX__`/`PX__` feature columns. It does not establish that a correctly aligned NumPy `X` or trained model already exists.

This notebook converts that completed output into an identity-preserving sequence bundle, checks the historical population contract, and freezes or reuses a patient-level TRAIN/VALIDATION/TEST manifest. It does not recreate `tensor_initialization.ipynb`, claims SQL, code mappings or the cohort. That upstream notebook was not available for inspection here.

**Expected source:** historical V63, claims vintage `20260825`, raw nonnegative feature counts. The documented full cohort is 23,151 snapshots, 12,447 patients and 1,345 positive snapshots; 12 months and 1,028 features (42 RX + 490 DX + 496 PX). `TIME_STEP=0` is newest and 11 oldest. Zero-activity months remain present. `RESP` and `TIME_STEP_MONTH` are not predictors.

**Outputs, kept locally:** a NumPy bundle with ordered keys/features/timesteps, a frozen manifest, and an aggregate split summary plus fingerprint. Notebook displays contain aggregate information only.

## 1. Read the local configuration

Copy `config.example.json` to ignored `config.local.json` at the repository root and edit it on the work laptop. Alternatively set `TAK861_CONFIG` to an approved local JSON configuration path. Relative artifact paths resolve against the configuration file, not the notebook location.

Complete the upstream review using the actual source implementation before setting its review flags to `true`. The flags confirm feature order, snapshot semantics, predictor cutoff and exclusion of outcome-window events; they are human attestations, not facts inferred from tensor values. Provide a local `review_reference`. A missing review deliberately stops the pipeline.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import Markdown, display

ROOT = next((candidate for candidate in (Path.cwd(), *Path.cwd().parents)
             if (candidate / "targeting_evaluation.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Launch this notebook from the repository directory or one of its child directories.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.configuration import load_config

cfg = load_config()

In [ ]:
from src.data_utils import (
    bundle_from_monthly,
    freeze_patient_split,
    load_bundle,
    save_bundle,
    split_summary,
    validate_population,
    validate_source_review,
)
from targeting_evaluation import manifest_fingerprint

validate_source_review(cfg["source_review"])

## 2. Load or export the existing completed monthly data

Supported `input.mode` values are `monthly_parquet`, `in_memory` and `bundle`. A complete bundle already present at `bundle_dir` is loaded and checked before any monthly table is collected. An incomplete or mismatched bundle fails; it is not silently overwritten.

For `in_memory`, run in a kernel where the existing `tensor_complete` variable is available as a Spark or pandas DataFrame. Spark `.toPandas()` collects the completed monthly table to the driver. The float32 `X` array alone is about 1.06 GiB for the documented population; the monthly DataFrame, conversion arrays and copies require substantially more memory. Use a suitably sized driver. `monthly_parquet` can read an approved Parquet file or Spark Parquet directory without repeating upstream feature work.

Use the original `feature_list` when it is present in the in-memory session, or configure `input.feature_order_json` containing the exact feature-name list. Otherwise the adapter uses the sorted prefixed feature names, matching the order described in the shared reference. Validate that convention locally. It never adds mappings, imputes missing month rows, drops malformed snapshots or converts numeric patient IDs into apparently valid strings.

In [ ]:
bundle_dir = cfg["bundle_dir"]
input_cfg = cfg["input"]
input_mode = input_cfg["mode"]

if bundle_dir.exists():
    bundle = load_bundle(bundle_dir)
    bundle_status = "Reused an existing complete bundle after integrity validation."
elif input_mode == "bundle":
    raise FileNotFoundError("Configured bundle does not exist. Export the completed monthly data first.")
else:
    if input_mode == "in_memory":
        if "tensor_complete" not in globals():
            raise RuntimeError("The existing tensor_complete DataFrame must be available in this kernel.")
        if isinstance(tensor_complete, pd.DataFrame):
            monthly_frame = tensor_complete
        elif callable(getattr(tensor_complete, "toPandas", None)):
            monthly_frame = tensor_complete.toPandas()
        else:
            raise TypeError("tensor_complete must be a pandas or Spark DataFrame.")
    elif input_mode == "monthly_parquet":
        if not input_cfg.get("path"):
            raise ValueError("Set input.path to the approved completed monthly Parquet source.")
        monthly_frame = pd.read_parquet(input_cfg["path"])
    else:
        raise ValueError("input.mode must be monthly_parquet, in_memory or bundle.")

    feature_names = None
    if input_cfg.get("feature_order_json"):
        with input_cfg["feature_order_json"].open(encoding="utf-8") as handle:
            feature_names = json.load(handle)
        if not isinstance(feature_names, list):
            raise TypeError("feature_order_json must contain a JSON list of feature names.")
    elif input_mode == "in_memory" and "feature_list" in globals():
        feature_names = list(feature_list)

    bundle = bundle_from_monthly(
        monthly_frame,
        feature_names=feature_names,
        expected_steps=cfg["expected"]["shape"][1],
    )
    validate_population(bundle, cfg["expected"])
    save_bundle(bundle, bundle_dir, source_review=cfg["source_review"])
    del monthly_frame
    bundle_status = "Created the identity-preserving bundle from the existing completed monthly data."

validate_population(bundle, cfg["expected"])
display(Markdown(bundle_status))

The adapter orders snapshots by both keys, orders each sequence by `TIME_STEP` from 0 through 11, and binds the feature axis to its saved vocabulary. Duplicate month rows, incomplete sequences, inconsistent within-snapshot labels, missing keys and nonfinite or negative counts fail validation. `TIME_STEP_MONTH` may be present as an unused source column; only the validated prefixed feature columns enter `X`.

Population validation compares the actual tensor shape, patient/positive counts and RX/DX/PX feature counts with the explicit local configuration. Investigate any discrepancy instead of relaxing the expected contract to make a run pass.

In [ ]:
population_summary = pd.DataFrame([{
    "n_snapshots": bundle.X.shape[0],
    "n_months": bundle.X.shape[1],
    "n_features": bundle.X.shape[2],
    "n_patients": bundle.snapshots["PATIENT_ID"].nunique(),
    "n_resp1": int(bundle.y.sum()),
    "response_rate": float(bundle.y.mean()),
    "RX_features": sum(name.startswith("RX__") for name in bundle.feature_names),
    "DX_features": sum(name.startswith("DX__") for name in bundle.feature_names),
    "PX_features": sum(name.startswith("PX__") for name in bundle.feature_names),
}])
display(population_summary)

## 3. Freeze or reuse the patient-level split

If the configured manifest already exists, reuse its exact assignments and verify that it covers this bundle with matching keys and labels. Configuration changes to fractions or seed do not replace an existing manifest.

Only when no manifest exists, split unique patients using the configured 70%/15%/15% defaults and seed 42, stratified by whether each patient has **any** positive snapshot. Then assign every snapshot for that patient to the same split. Stratification preserves patient groups without forcing a 1:1 outcome balance. Actual snapshot fractions can differ from patient fractions because patients contribute different numbers of snapshots.

This split is created before model fitting. For an already-trained model, supply its original manifest; a new split cannot retrospectively make that model's predictions held out.

In [ ]:
manifest_path = cfg["manifest_path"]
manifest_existed = manifest_path.exists()
manifest = freeze_patient_split(bundle, manifest_path, **cfg["split"])
summary = split_summary(manifest)
manifest_sha256 = manifest_fingerprint(manifest)

assert summary["n_snapshots"].sum() == bundle.X.shape[0]
assert summary["n_patients"].sum() == bundle.snapshots["PATIENT_ID"].nunique()
assert summary["n_resp1"].sum() == int(bundle.y.sum())

display(summary)
display(Markdown(
    "**Split status:** " + ("Reused the original manifest." if manifest_existed else "Created and froze the patient-level manifest.")
))

## 4. Save the aggregate split record

The manifest and bundle contain patient-level data and stay in the configured approved artifact location, outside version control. This final cell writes only aggregate split counts and a manifest fingerprint beside the manifest. The fingerprint binds identities, outcomes and assignments; it is an integrity check, not proof of upstream leakage prevention.

An existing audit must match the frozen manifest fingerprint and is preserved unchanged, including its original creation seed and fractions. When reusing a manifest without a prior audit, creation settings remain explicitly unknown; the current configuration is not attributed to that older split.

In [ ]:
summary_path = manifest_path.with_name(manifest_path.stem + "_summary.csv")
audit_path = manifest_path.with_name(manifest_path.stem + "_audit.json")

if audit_path.exists():
    with audit_path.open(encoding="utf-8") as handle:
        split_audit = json.load(handle)
    if not isinstance(split_audit, dict) or split_audit.get("snapshot_manifest_sha256") != manifest_sha256:
        raise ValueError("Existing split audit does not match the frozen manifest; investigate the discrepancy.")
    # Preserve the original creation settings and all other audit content exactly.
    # A later configuration file does not establish how a reused split was created.
    audit_status = "Verified and preserved the original split audit."
else:
    split_audit = {
        "snapshot_manifest_sha256": manifest_sha256,
        "patient_disjointness_checked": True,
        "exact_bundle_snapshot_match_checked": True,
        "manifest_created_with_this_audit": not manifest_existed,
        "split_configuration_used_for_creation": cfg["split"] if not manifest_existed else None,
        "creation_configuration_status": (
            "Recorded when this notebook created the split."
            if not manifest_existed else
            "Unknown: the frozen manifest already existed when this audit was first written."
        ),
        "snapshot_unit": "PATIENT_ID + END_DT",
        "source_review_is_local_attestation": True,
    }
    with audit_path.open("x", encoding="utf-8") as handle:
        handle.write(json.dumps(split_audit, indent=2, allow_nan=False) + "\n")
    audit_status = "Created the split audit without inventing prior creation settings."

summary.to_csv(summary_path, index=False)
display(Markdown(audit_status + " Saved the aggregate split summary locally. Continue to **02_transformer_training.ipynb**."))